# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### Simple Chain

In [2]:
from langchain_core.prompts import PromptTemplate  # prompt chain 구성
from langchain.chat_models import init_chat_model  # 모델 chain 구성 래퍼 
from langchain_core.output_parsers import StrOutputParser # 답변 문자형 변환

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?') 
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 특히 강릉·평창·홍천 지역의 감자
- **옥수수**: 찰옥수수로 유명한 홍천·정선 등
- **황태**: 인제 용대리 황태
- **한우**: 횡성한우
- **오징어·명태**: 동해안 지역의 수산물
- **더덕·산나물**: 양구, 횡성, 평창 등 산간 지역
- **곤드레**: 정선 곤드레
- **메밀 음식**: 평창·봉평 메밀과 메밀국수
- **도루묵·양미리**: 속초·양양 등 동해안 특산 수산물

그중에서도 **감자, 옥수수, 황태, 횡성한우, 정선 곤드레, 평창 메밀**이 특히 잘 알려져 있습니다.


### Sequential Chain

In [3]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm

eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""

print(chain1.invoke(eng_text))

chain2 = prompt2 | llm | output_parser

kor_text = """
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""

print(chain2.invoke(kor_text))

content='LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다(예: 특정 문서나 이메일에 대한 접근 권한이 없는 경우). 이러한 한계는 LLM이 특정 외부 데이터에 접근할 수 있도록 함으로써 해결할 수 있습니다.\n\n이를 위해서는 먼저 문서 로더(document loader)를 사용하여 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 97, 'total_tokens': 213, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI93tMDSxyhLGlK66kfzAla1DPL2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a04096-fa07-70a2-b622-fd94a145100f-0' tool_calls=[] invalid_tool_calls=[] usage_me

In [4]:
# Sequential chain
chain = chain1 | chain2
print(chain.invoke({'eng_text': eng_text}))

LLM은 특정 문서나 이메일 같은 외부 맥락에 직접 접근하기 어렵다는 한계가 있습니다. 이를 보완하려면 외부 데이터를 LLM에 연결해야 하며, LangChain에서는 문서 로더를 통해 PDF, 이메일, 웹사이트, YouTube 동영상 등 다양한 자료를 불러올 수 있습니다.


### Conditional Chain

In [5]:
from langchain_core.runnables import RunnableBranch # 조건에 따라 체인을 분기 실행해주는 Runnable

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)과 함께 작성해주세요.{question}')
math_chain = math_prompt | llm | output_parser

default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요.{question}')
default_chain = default_prompt | llm | output_parser

# math_chain 선택 함수 (질문에 계산 또는 calc가 포함되면 수학 체인 선택)
def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '')  # 입력받은 dict에서 question 키의 값을 추출(없으면 빈 문자열)
    return '계산' in question or 'calc' in question

# 분기 체인
branch_chain = RunnableBranch(
    (is_math_question, math_chain), # True/False 결과 조건이 True면 math_chain
    default_chain                   # False면 default_chain
)

branch_chain.invoke({'question': '125*3 + 50 계산해줘'})

'계산식은 다음과 같습니다.\n\n\\[\n125 \\times 3 + 50\n\\]\n\n먼저 곱셈을 계산합니다.\n\n\\[\n125 \\times 3 = 375\n\\]\n\n그다음 50을 더합니다.\n\n\\[\n375 + 50 = 425\n\\]\n\n따라서 답은\n\n\\[\n\\boxed{425}\n\\]\n\n입니다.'

In [6]:
print(branch_chain.invoke({'question': '나 오늘 우울해. 빵? 밥?'}))

오늘은 **따뜻한 밥** 어때? 🍚  
우울한 날엔 속을 편안하게 해주는 밥에 계란 하나, 김이나 국까지 곁들이면 조금 든든해질 거야.

근데 입맛이 거의 없으면 **달달한 빵과 따뜻한 음료**도 좋아. 오늘은 잘 먹는 것만으로도 충분히 잘하고 있는 거야.  
무슨 일 때문에 마음이 가라앉았어?


### Memory chain

'runnableWithMessageHistory'를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [7]:
from langchain_core.chat_history import BaseChatMessageHistory  # LANGCHAIN 대화기록 메모리 저장용 클래스
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage # 메시지 타입들
from pydantic import BaseModel, Field # Pydantic 모델(검증/기본값 생성) 도구
from typing import List # 타입 힌트(List)

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    # Field(default_factoru=list) : 인스턴스마다 독립적인 message list를 구성
    messages: List[BaseMessage] = Field(default_factory=list)

    def add_messages(self, messages: List[BaseMessage]) -> None:
        self.messages.extend(messages) # 전달받은 메시지들을 기존 리스트 뒤에 추가

    def clear(self) -> None:
        self.messages = []  # 저장된 메시지들을 초기화


store = {} # {session_id: 히스토리 객체()} 저장소

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryHistory() # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[session_id] # 해당 세션의 히스토리 객체 반환


history1 = get_by_session_id("1") # 세션 ID '1'의 히스토리 가져오기 (없으면 메모리 공간 생성)
history1.add_messages([AIMessage(content="반갑습니다. Capybara님!")]) # AI 메시지 추가
history1.add_messages([HumanMessage(content="그래~ 나 Cap이야~ 만나서 반갑다!!")]) # 유저메시지 추가
print(f"{history1 = }") # f"history1 = {history1}"

history2 = get_by_session_id("2") # 세션 ID '2'의 히스토리 가져오기
print(f"{history2 = }")

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. Capybara님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래~ 나 Cap이야~ 만나서 반갑다!!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,              # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question',  # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key='history'  # 프롬프트에서 history 받을 변수명
)

chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지를 3마리 키우고 있습니다.'
}, config={  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '100' # 
    }
})



c:\Users\SJ\OneDrive\Desktop\study\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='그렇군요. 민수는 강아지를 3마리 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 38, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 31, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI9q6LxggOyJQiIUs8DjXAqdfDMB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04097-bef4-77a2-956a-c7e1c5205c90-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 61, 'total_tokens': 99, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 31}})

In [9]:

chain_with_history.invoke({
    'domain': 'math',
    'question': '소라는 고양이를 4마리 키우고 있습니다.'
}, config={  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '100' # 
    }
})

AIMessage(content='그렇군요. 소라는 고양이를 4마리 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 83, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 57, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI9trhjHXFUlsEmyV9oSYANKmTwt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04097-cae3-7ea0-87fa-c94ba6555564-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 86, 'total_tokens': 169, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 57}})

### ChatMessageHistory

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm | output_parser

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,              # session ID로 히스토리 객체를 가져오는 함수
    input_messages_key='question',  # question 키의 값은 사용자 메시지
    history_messages_key='history'  # 프롬프트에서 히스토리를 받을 변수명
)

chain_with_history.invoke({
    'domain': '심리상담',
    'question': '더워서 짜증나네.'
}, config={
    'configurable': {
        'session_id': '200'
    }
})

chain_with_history.invoke({
    'domain': '심리상담',
    'question': '더워서 짜증나네.'
}, config={  # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '200' # 
    }
})


c:\Users\SJ\OneDrive\Desktop\study\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'맞아요, 더우면 몸이 축 처지고 괜히 예민해지죠. 시원한 물 조금씩 마시고, 목·겨드랑이·손목을 차갑게 식히거나 서늘한 곳에서 잠깐 쉬어보세요.  \n오늘은 가능한 한 무리하지 말고, 짜증나는 감정도 “지금 더워서 그렇다” 하고 잠시 흘려보내도 괜찮아요.'

In [12]:
chain_with_history.invoke({
    'domain': '심리상담',
    'question': '그럼 니가 날씨를 좋게 만들어주면 되잖아 AI니까 가능하잖아'
}, config = {
    'configurable': {
        'session_id': '200'
    }
})

'그러고 싶지만, 아쉽게도 제가 실제 날씨를 바꿀 수는 없어요. AI라고 하늘까지 조종하진 못하네요 😅\n\n대신 지금 체감온도를 낮추는 건 도와드릴 수 있어요:  \n- 찬물이나 얼음물 조금씩 마시기  \n- 목·손목·겨드랑이를 시원하게 식히기  \n- 커튼을 치고 선풍기를 창문 쪽으로 틀어 더운 공기 빼기  \n- 가능하면 샤워 후 얇은 옷 입고 잠깐 쉬기  \n\n일단 오늘 더위는 제가 같이 욕해드릴게요. 진짜 너무 덥네요.'

In [13]:
store

{'200': InMemoryChatMessageHistory(messages=[HumanMessage(content='더워서 짜증나네.', additional_kwargs={}, response_metadata={}), AIMessage(content='날씨가 더우면 몸도 쉽게 지치고 사소한 일에도 예민해질 수 있어요. 우선 시원한 물을 마시고, 에어컨이나 선풍기를 켜고, 가능하면 잠깐 서늘한 곳에서 쉬어보세요.  \n\n지금 특히 짜증나게 만든 일이 따로 있나요, 아니면 더위 자체가 너무 힘든가요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='더워서 짜증나네.', additional_kwargs={}, response_metadata={}), AIMessage(content='맞아요, 더우면 몸이 축 처지고 괜히 예민해지죠. 시원한 물 조금씩 마시고, 목·겨드랑이·손목을 차갑게 식히거나 서늘한 곳에서 잠깐 쉬어보세요.  \n오늘은 가능한 한 무리하지 말고, 짜증나는 감정도 “지금 더워서 그렇다” 하고 잠시 흘려보내도 괜찮아요.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그럼 니가 날씨를 좋게 만들어주면 되잖아 AI니까 가능하잖아', additional_kwargs={}, response_metadata={}), AIMessage(content='그러고 싶지만, 아쉽게도 제가 실제 날씨를 바꿀 수는 없어요. AI라고 하늘까지 조종하진 못하네요 😅\n\n대신 지금 체감온도를 낮추는 건 도와드릴 수 있어요:  \n- 찬물이나 얼음물 조금씩 마시기  \n- 목·손목·겨드랑이를 시원하게 식히기  \n- 커튼을 치고 선풍기를 창문 쪽으로 틀어 더운 공기 빼기  \n- 가능하면 

##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리